# 05 — Weight-level learning: train a neural policy without PyTorch

You asked what it means to **actually train model weights**.

Here we make a tiny neural network ourselves using only NumPy.

First, generate demonstrations from the readable Elder Mist rule policy.

Then train:

```text
battle observation
      ↓
Linear weights W1
      ↓
ReLU
      ↓
Linear weights W2
      ↓
action probabilities
```

This is **behavior cloning**, not RL.

That distinction is deliberate.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo))

from eldermist import ElderMistReplicaEnv, ACTIONS
from eldermist.policies import sensible_rule_policy
from eldermist.mlp_clone import TinyMLP


## 1. Collect `(observation, action)` examples

This is supervised imitation:

```text
X = what policy saw
y = action demonstrated
```


In [ ]:
X, y = [], []

for seed in range(250):
    env = ElderMistReplicaEnv()
    obs, _ = env.reset(seed=seed)

    while True:
        action = sensible_rule_policy(env, obs)
        X.append(obs.copy())
        y.append(action)

        obs, _, term, trunc, _ = env.step(action)
        if term or trunc:
            break

X = np.asarray(X, dtype=np.float64)
y = np.asarray(y, dtype=np.int64)

print("examples:", len(X))
print("input dimensions:", X.shape[1])
print("possible actions:", len(ACTIONS))


## 2. What is a dimension?

If one observation is:

```text
[0.82, 1.0, 0.62, 0.70, ...]
```

each position is one **dimension** of the input vector.

The network maps that point in input space into hidden activations.

Those hidden activations are a **learned representation** because training determines what numerical patterns become useful internally.


In [ ]:
rng = np.random.default_rng(0)
perm = rng.permutation(len(X))
split = int(0.8 * len(X))
train_i, test_i = perm[:split], perm[split:]

model = TinyMLP(
    in_dim=X.shape[1],
    hidden_dim=32,
    out_dim=len(ACTIONS),
    seed=0,
)

before = model.W1.copy()
print("First five W1 values BEFORE training:")
print(before.ravel()[:5])


## 3. Backpropagation

Inside `train_step()`:

```python
# forward pass
z1 = X @ W1 + b1
h = ReLU(z1)
logits = h @ W2 + b2

# loss says how wrong we were
loss = cross_entropy(...)

# backward pass computes gradients
dW2 = ...
dW1 = ...

# optimizer step changes weights
W1 -= learning_rate * dW1
W2 -= learning_rate * dW2
```

That final subtraction is **weight-level learning**.

No prompt changed.

No RAG document changed.

The numeric parameters inside the network changed.


In [ ]:
losses = []

for epoch in range(120):
    # Simple full-batch training so every line is easy to inspect.
    loss = model.train_step(X[train_i], y[train_i], lr=0.05)
    losses.append(loss)

    if epoch % 20 == 0:
        pred = model.predict(X[test_i])
        acc = (pred == y[test_i]).mean()
        print(f"epoch={epoch:03d} loss={loss:.4f} held-out imitation accuracy={acc:.3f}")


In [ ]:
after = model.W1

print("First five W1 values AFTER training:")
print(after.ravel()[:5])

print("\nAbsolute amount those five weights changed:")
print(np.abs(after.ravel()[:5] - before.ravel()[:5]))

plt.figure(figsize=(8,4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Weight-level learning")
plt.show()


## 4. Why this is not RL

We trained with:

```text
"the demonstrated correct action was SUNBALL"
```

That is behavior cloning.

RL instead gives:

```text
you chose SUNBALL
↓
environment changed
↓
eventually won / lost / got return
↓
update policy to increase expected future return
```

Both can update neural weights.

The difference is the **training signal**.

This is why:

```text
fine-tuning vs RL
```

is not the same distinction as:

```text
policy vs not-policy
```

A fine-tuned model can be a policy.
A prompted model can be a policy.
A Q-table can be a policy.
An RL-trained network can be a policy.


## 5. OpenAI Five bridge

The conceptual jump to Dota/OpenAI Five is:

```text
our tiny numeric battle state
        ↓
tiny MLP policy

becomes

large partially-observable Dota observations + memory
        ↓
large recurrent neural policy/value networks
        ↓
millions of self-play experiences
        ↓
policy-gradient/PPO-style weight updates
```

Same broad idea. Vastly larger scale.
